# Multi3Hate — Q1-Ready Verification Notebook (Kaggle version)
Cultural-Bias Debiasing: corrected pipeline, checkpoint-verified evaluation,
FairPIVARA comparison, statistical testing, power analysis, and a reusable
reproducibility diagnostic toolkit.

**Before running, in the Kaggle notebook Settings panel (right sidebar):**
1. **Accelerator -> GPU T4 x2** (or P100) -- this notebook needs a GPU.
2. **Internet -> On** -- required for `pip install`, the `git clone`, and
   downloading OpenCLIP's pretrained weights.

**Checkpoints:** this notebook loads two pre-trained checkpoints
(`adversarial_debiased_model_best.pt`, `baseline_model_best.pt`). Kaggle
notebooks don't support interactive file uploads mid-run the way Colab
does -- attach them via **Add Data -> Upload -> New Dataset** (upload both
`.pt` files as one dataset) before running this notebook, or drag them into
an existing attached dataset. The checkpoint-loading cell searches
`/kaggle/input/**/*.pt` automatically; if it can't find them, it will print
a loud warning and train fresh models instead, which will not match
previously reported numbers.


In [1]:
#@title 0. Check Kaggle inputs and GPU
import os

print("Attached Kaggle input datasets (should include your checkpoint upload):")
if os.path.exists("/kaggle/input"):
    for name in sorted(os.listdir("/kaggle/input")):
        print(" -", name)
else:
    print("  /kaggle/input does not exist -- no datasets attached yet.")

import torch
print("\nGPU available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("\u26a0\ufe0f  No GPU detected -- set Settings > Accelerator > GPU T4 x2 "
          "and restart the session before continuing.")


Attached Kaggle input datasets (should include your checkpoint upload):
 - datasets

GPU available: True


In [2]:
#@title 1. Install only the dependencies needed for this notebook
# Kaggle: run this cell first (ensure Internet is ON in Settings).

!pip install -q open-clip-torch captum scikit-learn pandas matplotlib pillow tqdm

import os
import sys
import copy
import random
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.amp import autocast, GradScaler
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.utils.class_weight import compute_class_weight
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 22.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.6 MB/s eta 0:00:00
PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


In [3]:
#@title 2. Reproducibility + paths

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BASE_DIR = Path("/kaggle/working/Multi3Hate")
MEME_DIR = BASE_DIR / "data" / "memes"
CAPTIONS_DIR = BASE_DIR / "data" / "captions"
RESULTS_DIR = BASE_DIR / "results"
FIGURES_DIR = BASE_DIR / "figures"
MODELS_DIR = BASE_DIR / "models"

for p in [BASE_DIR, RESULTS_DIR, FIGURES_DIR, MODELS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

DEBIAS_CHECKPOINT = MODELS_DIR / "adversarial_debiased_model_best.pt"
BASELINE_CHECKPOINT = MODELS_DIR / "baseline_model_best.pt"

print("Device:", device)
print("Base directory:", BASE_DIR)

Device: cuda
Base directory: /kaggle/working/Multi3Hate


## Environment pinning & full determinism
Required for a Q1 submission: document the exact environment this run used, and enable every determinism switch PyTorch offers, so a discrepancy against a future re-run can be diagnosed against a known, recorded baseline rather than guessed at.

In [4]:
#@title 2b. Environment pinning + full determinism
import subprocess
import sklearn
import open_clip as _open_clip_version_check

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
try:
    torch.use_deterministic_algorithms(True, warn_only=True)
except Exception as e:
    print("torch.use_deterministic_algorithms not fully supported here:", e)

env_versions = {
    "python": subprocess.run(["python", "--version"], capture_output=True, text=True).stdout.strip(),
    "torch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only",
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "sklearn": sklearn.__version__,
    "open_clip": _open_clip_version_check.__version__,
    "seed": SEED,
}
print("=" * 70)
print("ENVIRONMENT SNAPSHOT (record this alongside any reported result)")
print("=" * 70)
for k, v in env_versions.items():
    print(f"  {k}: {v}")

# Full pip freeze, saved for the paper's reproducibility appendix / supplementary material.
freeze_path = RESULTS_DIR / "environment_freeze.txt"
with open(freeze_path, "w") as f:
    subprocess.run(["pip", "freeze"], stdout=f)
print(f"\nFull pip freeze saved to: {freeze_path}")
print("Attach this file (or its checksum) alongside any reported numbers so a")
print("future re-run can diff environments directly instead of guessing.")


ENVIRONMENT SNAPSHOT (record this alongside any reported result)
  python: Python 3.12.13
  torch: 2.10.0+cu128
  cuda_available: True
  gpu: Tesla T4
  numpy: 2.0.2
  pandas: 2.3.3
  sklearn: 1.6.1
  open_clip: 3.3.0
  seed: 42

Full pip freeze saved to: /kaggle/working/Multi3Hate/results/environment_freeze.txt
Attach this file (or its checksum) alongside any reported numbers so a
future re-run can diff environments directly instead of guessing.


In [5]:
import subprocess
import shutil # Added import

if not (BASE_DIR / ".git").exists():
    print("Cloning Multi3Hate repository...")
    # The directory is created above. Clone into it only if it is not already a git repo.
    # If BASE_DIR exists and is not a git repo, remove it to allow cloning.
    if BASE_DIR.is_dir(): # Check if it's a directory
        shutil.rmtree(BASE_DIR) # Remove the directory and its contents

    subprocess.run(
        ["git", "clone", "https://github.com/MinhDucBui/Multi3Hate.git", str(BASE_DIR)],
        check=True
    )
else:
    print("Repository already exists.")

# Re-create output folders because cloning may have created the repository tree.
for p in [RESULTS_DIR, FIGURES_DIR, MODELS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Repository:", BASE_DIR)
print("MEME_DIR exists:", MEME_DIR.exists())
print("CAPTIONS_DIR exists:", CAPTIONS_DIR.exists())

Cloning Multi3Hate repository...


Cloning into '/kaggle/working/Multi3Hate'...


Repository: /kaggle/working/Multi3Hate
MEME_DIR exists: True
CAPTIONS_DIR exists: True


## Corrected data construction
Uses real per-culture captions (`captions/{lang}.csv`) and real majority-voted labels (`final_annotations.csv`), joined against image paths resolved by scanning disk directly rather than reconstructing filenames from `meme_id` (the latter was the original root-cause bug).

In [6]:
#@title 4. Build the corrected real-image + real-caption dataset

CULTURE_TO_LANG = {
    "US": "en",
    "DE": "de",
    "MX": "es",
    "CN": "zh",
    "IN": "hi",
}

# ---- Build meme_id -> template folder from the actual disk layout ----
en_dir = MEME_DIR / "en"
if not en_dir.exists():
    raise FileNotFoundError(
        f"Expected image directory not found: {en_dir}. "
        "Check that the Multi3Hate repository/data is available in this Kaggle runtime."
    )

folder_lookup = {}
for template_folder in os.listdir(en_dir):
    template_path = en_dir / template_folder
    if not template_path.is_dir():
        continue
    for fname in os.listdir(template_path):
        if fname.lower().endswith(".jpg"):
            stem = Path(fname).stem
            try:
                folder_lookup[int(stem)] = template_folder
            except ValueError:
                pass

print("Meme IDs mapped to template folders:", len(folder_lookup))

# ---- Load real per-language captions ----
caption_frames = []

for culture, lang in CULTURE_TO_LANG.items():
    caption_file = CAPTIONS_DIR / f"{lang}.csv"
    if not caption_file.exists():
        raise FileNotFoundError(f"Caption file not found: {caption_file}")

    df = pd.read_csv(caption_file)
    required = {"Meme ID", "Translation"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{caption_file} is missing columns: {sorted(missing)}")

    df["culture"] = culture
    df["language"] = lang
    df["clean_text"] = (
        df["Translation"]
        .fillna("")
        .astype(str)
        .str.replace("<sep>", " ", regex=False)
        .str.strip()
    )

    caption_frames.append(
        df[["Meme ID", "culture", "language", "clean_text"]]
    )

captions_long = (
    pd.concat(caption_frames, ignore_index=True)
    .rename(columns={"Meme ID": "meme_id"})
)

# ---- Load real majority-voted labels ----
final_annotations_path = BASE_DIR / "data" / "final_annotations.csv"
if not final_annotations_path.exists():
    raise FileNotFoundError(f"Missing final annotations: {final_annotations_path}")

final_annotations = pd.read_csv(final_annotations_path)

expected_culture_cols = list(CULTURE_TO_LANG.keys())
missing_culture_cols = [c for c in expected_culture_cols if c not in final_annotations.columns]
if missing_culture_cols:
    raise ValueError(
        f"final_annotations.csv does not contain expected culture columns: {missing_culture_cols}"
    )

labels_long = (
    final_annotations
    .melt(
        id_vars="Meme ID",
        value_vars=expected_culture_cols,
        var_name="culture",
        value_name="label",
    )
    .rename(columns={"Meme ID": "meme_id"})
)

# ---- Join labels + captions + real image path ----
aggregated_annotations = labels_long.merge(
    captions_long,
    on=["meme_id", "culture"],
    how="left"
)

aggregated_annotations["template_folder"] = (
    aggregated_annotations["meme_id"].map(folder_lookup)
)

aggregated_annotations["image_path"] = aggregated_annotations.apply(
    lambda r: str(
        MEME_DIR
        / r["language"]
        / str(r["template_folder"])
        / f"{int(r['meme_id'])}.jpg"
    ),
    axis=1,
)

aggregated_annotations = aggregated_annotations.rename(
    columns={"clean_text": "original_text"}
)

# ---- Strict validation ----
n_missing_img = (~aggregated_annotations["image_path"].apply(os.path.exists)).sum()
n_empty_text = (
    aggregated_annotations["original_text"].fillna("").astype(str).str.len() == 0
).sum()

# The corrected notebook used this check to detect identical captions across cultures.
n_duplicate_text_groups = (
    aggregated_annotations.groupby("meme_id")["original_text"].nunique() == 1
).sum()

print(f"Rows: {len(aggregated_annotations)}")
print(f"Missing images: {n_missing_img}")
print(f"Empty captions: {n_empty_text}")
print(f"Memes with one unique caption across cultures: {n_duplicate_text_groups}")

assert n_missing_img == 0, "Missing real image paths detected."
assert n_empty_text == 0, "Empty captions detected."
assert n_duplicate_text_groups == 0, (
    "The corrected notebook's caption-validation condition failed. "
    "Do not proceed until the data are checked."
)

GLOBAL_CULTURE_TO_IDX = {
    c: i for i, c in enumerate(sorted(aggregated_annotations["culture"].unique()))
}

print("Culture mapping:", GLOBAL_CULTURE_TO_IDX)

Meme IDs mapped to template folders: 300
Rows: 1500
Missing images: 0
Empty captions: 0
Memes with one unique caption across cultures: 0
Culture mapping: {'CN': 0, 'DE': 1, 'IN': 2, 'MX': 3, 'US': 4}


In [7]:
#@title 5. Reproduce the grouped train/validation/test split

# Same split logic as the corrected section in the uploaded notebook.
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=SEED
)

train_idx, temp_idx = next(
    gss.split(
        aggregated_annotations,
        groups=aggregated_annotations["meme_id"]
    )
)

train_data = aggregated_annotations.iloc[train_idx].reset_index(drop=True)
temp_data = aggregated_annotations.iloc[temp_idx].reset_index(drop=True)

gss_val = GroupShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=SEED
)

val_idx, test_idx = next(
    gss_val.split(
        temp_data,
        groups=temp_data["meme_id"]
    )
)

val_data = temp_data.iloc[val_idx].reset_index(drop=True)
test_data = temp_data.iloc[test_idx].reset_index(drop=True)

assert set(train_data["meme_id"]).isdisjoint(set(val_data["meme_id"]))
assert set(train_data["meme_id"]).isdisjoint(set(test_data["meme_id"]))
assert set(val_data["meme_id"]).isdisjoint(set(test_data["meme_id"]))

print(f"Train: {len(train_data)} rows / {train_data['meme_id'].nunique()} unique memes")
print(f"Val:   {len(val_data)} rows / {val_data['meme_id'].nunique()} unique memes")
print(f"Test:  {len(test_data)} rows / {test_data['meme_id'].nunique()} unique memes")

Train: 1050 rows / 210 unique memes
Val:   225 rows / 45 unique memes
Test:  225 rows / 45 unique memes


In [8]:
#@title 6. Dataset + DataLoader

import open_clip

# Use the SAME OpenCLIP preprocessing used by the uploaded notebook.
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32",
    pretrained="laion2b_s34b_b79k"
)
clip_model = clip_model.to(device).eval()

for p in clip_model.parameters():
    p.requires_grad = False

clip_tokenizer = open_clip.get_tokenizer("ViT-B-32")

CLIP_FEATURE_DIM = 512
FUSED_FEATURE_DIM = 1024

print("OpenCLIP loaded.")
print("Image feature dimension:", CLIP_FEATURE_DIM)
print("Fused feature dimension:", FUSED_FEATURE_DIM)


class Multi3HateDataset(Dataset):
    def __init__(self, dataframe):
        self.annotations = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, idx):
        row = self.annotations.iloc[idx]

        image = Image.open(row["image_path"]).convert("RGB")
        image = clip_preprocess(image)

        return {
            "image": image,
            "text": str(row["original_text"]),
            "label": torch.tensor(int(row["label"]), dtype=torch.long),
            "culture": row["culture"],
            "culture_idx": torch.tensor(
                GLOBAL_CULTURE_TO_IDX[row["culture"]],
                dtype=torch.long
            ),
            "meme_id": int(row["meme_id"]),
            "language": row["language"],
            "image_path": row["image_path"],
        }


def collate_fn(batch):
    return {
        "image": torch.stack([x["image"] for x in batch]),
        "text": [x["text"] for x in batch],
        "label": torch.stack([x["label"] for x in batch]),
        "culture": [x["culture"] for x in batch],
        "culture_idx": torch.stack([x["culture_idx"] for x in batch]),
        "meme_id": [x["meme_id"] for x in batch],
        "language": [x["language"] for x in batch],
        "image_path": [x["image_path"] for x in batch],
    }


BATCH_SIZE = 16
NUM_WORKERS = 2  # Kaggle's disk I/O benefits from a couple of workers; drop to 0 if you see worker crashes.
PIN_MEMORY = torch.cuda.is_available()

train_dataset = Multi3HateDataset(train_data)
val_dataset = Multi3HateDataset(val_data)
test_dataset = Multi3HateDataset(test_data)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, collate_fn=collate_fn
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, collate_fn=collate_fn
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, collate_fn=collate_fn
)

print("DataLoaders ready.")

open_clip_model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

OpenCLIP loaded.
Image feature dimension: 512
Fused feature dimension: 1024
DataLoaders ready.


In [9]:
#@title 7. CLIP feature extraction helper

@torch.no_grad()
def get_clip_features(images, texts):
    images = images.to(device, non_blocking=True)
    text_tokens = clip_tokenizer(texts).to(device)

    image_features = clip_model.encode_image(images)
    text_features = clip_model.encode_text(text_tokens)

    image_features = F.normalize(image_features, p=2, dim=-1)
    text_features = F.normalize(text_features, p=2, dim=-1)

    return torch.cat([image_features, text_features], dim=1)


# Sanity check on one batch
batch = next(iter(test_loader))
features = get_clip_features(batch["image"], batch["text"])

print("Feature shape:", tuple(features.shape))
print("Expected second dimension:", FUSED_FEATURE_DIM)
assert features.shape[1] == FUSED_FEATURE_DIM

Feature shape: (16, 1024)
Expected second dimension: 1024


## Diagnostic: feature distinctness
Confirms the fix actually produces distinct per-culture inputs -- this is the check that originally caught the constant-classifier collapse (5/16 distinct feature vectors under the broken pipeline vs. 16/16 after the fix).

In [10]:
#@title Diagnostic A. Feature distinctness (re-verify the root-cause fix)
sample_batch = next(iter(test_loader))
with torch.no_grad():
    diag_features = get_clip_features(sample_batch["image"], sample_batch["text"])

feats_cpu = diag_features.detach().cpu()
n_total = feats_cpu.shape[0]
n_distinct = torch.unique(feats_cpu, dim=0).shape[0]
feats_norm = feats_cpu / feats_cpu.norm(dim=1, keepdim=True).clamp_min(1e-8)
cos_sim = feats_norm @ feats_norm.T
off_diag = ~torch.eye(n_total, dtype=torch.bool)

print("=" * 60)
print("FEATURE DISTINCTNESS CHECK")
print("=" * 60)
print(f"Batch size               : {n_total}")
print(f"Distinct feature vectors : {n_distinct} / {n_total}")
print(f"Mean pairwise cosine sim : {cos_sim[off_diag].mean().item():.4f}")
print(f"Max pairwise cosine sim  : {cos_sim[off_diag].max().item():.4f}")

if n_distinct < n_total:
    print("\n\u26a0\ufe0f FAIL: duplicate feature vectors found -- the root-cause bug may "
          "have resurfaced. Do not trust downstream results.")
else:
    print("\n\u2705 PASS: all feature vectors distinct.")


FEATURE DISTINCTNESS CHECK
Batch size               : 16
Distinct feature vectors : 16 / 16
Mean pairwise cosine sim : 0.4482
Max pairwise cosine sim  : 0.6018

✅ PASS: all feature vectors distinct.


In [11]:
#@title 8. Model definitions used by the original final comparison

def compute_class_weights_tensor(labels, num_classes=2):
    labels = np.asarray(labels)
    present = np.unique(labels)

    weights = np.ones(num_classes, dtype=np.float32)

    if len(present) > 1:
        balanced = compute_class_weight(
            class_weight="balanced",
            classes=present,
            y=labels
        )
        for c, w in zip(present, balanced):
            weights[int(c)] = w

    return torch.tensor(weights, dtype=torch.float32)


train_hate_class_weights = compute_class_weights_tensor(
    train_data["label"].values, num_classes=2
)

train_culture_labels = train_data["culture"].map(GLOBAL_CULTURE_TO_IDX).values
train_culture_class_weights = compute_class_weights_tensor(
    train_culture_labels,
    num_classes=len(GLOBAL_CULTURE_TO_IDX)
)

print("Hate class weights:", train_hate_class_weights.tolist())
print("Culture class weights:", train_culture_class_weights.tolist())


class GradientReversalFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambda_):
        ctx.lambda_ = lambda_
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambda_ * grad_output, None


class GradientReversalLayer(nn.Module):
    def __init__(self, lambda_=1.0):
        super().__init__()
        self.lambda_ = lambda_

    def forward(self, x):
        return GradientReversalFunction.apply(x, self.lambda_)


class AdversarialDebiasingModel(nn.Module):
    def __init__(
        self,
        feature_dim,
        num_classes=2,
        num_cultures=5,
        hidden_dim=512,
        dropout=0.3,
        lambda_adv=0.2
    ):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(feature_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_classes),
        )

        self.grl = GradientReversalLayer(lambda_adv)

        self.adversary = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_cultures),
        )

    def set_lambda(self, lambda_adv):
        self.grl.lambda_ = lambda_adv

    def forward(self, features, return_features=False):
        shared = self.encoder(features)
        hate_logits = self.classifier(shared)
        culture_logits = self.adversary(self.grl(shared))

        if return_features:
            return hate_logits, culture_logits, shared

        return hate_logits, culture_logits


class BaselineClassifier(nn.Module):
    def __init__(
        self,
        feature_dim,
        num_classes=2,
        hidden_dim=512,
        dropout=0.3
    ):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(feature_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_classes),
        )

    def forward(self, features):
        return self.classifier(self.encoder(features))


print("Model definitions ready.")

Hate class weights: [1.1513158082962036, 0.8838383555412292]
Culture class weights: [1.0, 1.0, 1.0, 1.0, 1.0]
Model definitions ready.


In [12]:
#@title 9. Training helpers

NUM_EPOCHS_DEBIASED = 30
PATIENCE_DEBIASED = 5
NUM_EPOCHS_BASELINE = 40
PATIENCE_BASELINE = 6

LEARNING_RATE = 1e-4
HIDDEN_DIM = 512
LAMBDA_ADV = 0.2
ACCUM_STEPS = 2


def scheduled_lambda(progress, lambda_max=LAMBDA_ADV):
    # Same sigmoid-style schedule used by the uploaded notebook.
    return lambda_max * (
        2.0 / (1.0 + np.exp(-10.0 * progress)) - 1.0
    )


def evaluate_debiased(model, loader):
    model.eval()

    all_preds, all_labels = [], []
    all_cultures = []
    total_loss = 0.0
    total_adv_loss = 0.0

    hate_loss_fn = nn.CrossEntropyLoss(weight=train_hate_class_weights.to(device))
    culture_loss_fn = nn.CrossEntropyLoss(weight=train_culture_class_weights.to(device))

    for batch in tqdm(loader, desc="Evaluating debiased", leave=False):
        features = get_clip_features(batch["image"], batch["text"])
        labels = batch["label"].to(device)
        cultures = batch["culture_idx"].to(device)

        hate_logits, culture_logits = model(features)

        total_loss += hate_loss_fn(hate_logits, labels).item()
        total_adv_loss += culture_loss_fn(culture_logits, cultures).item()

        all_preds.extend(hate_logits.argmax(dim=1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_cultures.extend(batch["culture"])

    culture_results = {}
    for culture in sorted(set(all_cultures)):
        mask = np.array(all_cultures) == culture
        culture_results[culture] = accuracy_score(
            np.array(all_labels)[mask],
            np.array(all_preds)[mask]
        )

    return {
        "hate_loss": total_loss / max(len(loader), 1),
        "adv_loss": total_adv_loss / max(len(loader), 1),
        "hate_acc": accuracy_score(all_labels, all_preds),
        "adv_acc": np.nan,
        "culture_results": culture_results,
        "predictions": all_preds,
        "labels": all_labels,
    }


def train_debiased(model, train_loader, val_loader, checkpoint_path):
    model = model.to(device)

    hate_loss_fn = nn.CrossEntropyLoss(
        weight=train_hate_class_weights.to(device)
    )
    culture_loss_fn = nn.CrossEntropyLoss(
        weight=train_culture_class_weights.to(device)
    )

    optimizer = AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=1e-4
    )

    use_amp = torch.cuda.is_available()
    scaler = GradScaler("cuda", enabled=use_amp)

    best_val_loss = float("inf")
    best_state = None
    no_improve = 0

    history = {
        "train_hate_loss": [],
        "train_adv_loss": [],
        "val_hate_loss": [],
        "val_hate_acc": [],
        "lambda_adv": [],
    }

    for epoch in range(NUM_EPOCHS_DEBIASED):
        model.train()
        optimizer.zero_grad()

        current_lambda = scheduled_lambda(
            epoch / max(NUM_EPOCHS_DEBIASED - 1, 1)
        )
        model.set_lambda(current_lambda)

        train_hate_loss = 0.0
        train_adv_loss = 0.0

        for i, batch in enumerate(
            tqdm(train_loader, desc=f"Debiased epoch {epoch+1}/{NUM_EPOCHS_DEBIASED}", leave=False)
        ):
            features = get_clip_features(batch["image"], batch["text"])
            hate_labels = batch["label"].to(device)
            culture_labels = batch["culture_idx"].to(device)

            with autocast("cuda", enabled=use_amp):
                hate_logits, culture_logits = model(features)

                hate_loss = hate_loss_fn(hate_logits, hate_labels)
                culture_loss = culture_loss_fn(culture_logits, culture_labels)

                combined_loss = hate_loss + culture_loss
                scaled_loss = combined_loss / ACCUM_STEPS

            scaler.scale(scaled_loss).backward()

            is_last = (i + 1) == len(train_loader)

            if (i + 1) % ACCUM_STEPS == 0 or is_last:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()

            train_hate_loss += hate_loss.item()
            train_adv_loss += culture_loss.item()

        val_metrics = evaluate_debiased(model, val_loader)

        avg_train_hate = train_hate_loss / max(len(train_loader), 1)
        avg_train_adv = train_adv_loss / max(len(train_loader), 1)

        history["train_hate_loss"].append(avg_train_hate)
        history["train_adv_loss"].append(avg_train_adv)
        history["val_hate_loss"].append(val_metrics["hate_loss"])
        history["val_hate_acc"].append(val_metrics["hate_acc"])
        history["lambda_adv"].append(current_lambda)

        print(
            f"Epoch {epoch+1:02d}: "
            f"train_hate={avg_train_hate:.4f} | "
            f"train_adv={avg_train_adv:.4f} | "
            f"val_hate_loss={val_metrics['hate_loss']:.4f} | "
            f"val_acc={val_metrics['hate_acc']:.3f} | "
            f"lambda={current_lambda:.4f}"
        )

        if val_metrics["hate_loss"] < best_val_loss - 1e-4:
            best_val_loss = val_metrics["hate_loss"]
            best_state = copy.deepcopy(model.state_dict())
            no_improve = 0
            torch.save(best_state, checkpoint_path)
        else:
            no_improve += 1
            if no_improve >= PATIENCE_DEBIASED:
                print(f"Early stopping at epoch {epoch+1}.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, history


def evaluate_baseline(model, loader):
    model.eval()

    all_preds, all_labels, all_cultures = [], [], []
    total_loss = 0.0

    loss_fn = nn.CrossEntropyLoss(
        weight=train_hate_class_weights.to(device)
    )

    for batch in tqdm(loader, desc="Evaluating baseline", leave=False):
        features = get_clip_features(batch["image"], batch["text"])
        labels = batch["label"].to(device)

        logits = model(features)
        total_loss += loss_fn(logits, labels).item()

        all_preds.extend(logits.argmax(dim=1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_cultures.extend(batch["culture"])

    culture_accs = {}
    for culture in sorted(set(all_cultures)):
        mask = np.array(all_cultures) == culture
        culture_accs[culture] = accuracy_score(
            np.array(all_labels)[mask],
            np.array(all_preds)[mask]
        )

    return {
        "hate_loss": total_loss / max(len(loader), 1),
        "overall_acc": accuracy_score(all_labels, all_preds),
        "culture_accs": culture_accs,
        "predictions": all_preds,
        "labels": all_labels,
    }


def train_baseline(model, train_loader, val_loader, checkpoint_path):
    model = model.to(device)

    loss_fn = nn.CrossEntropyLoss(
        weight=train_hate_class_weights.to(device)
    )

    optimizer = AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=1e-4
    )

    use_amp = torch.cuda.is_available()
    scaler = GradScaler("cuda", enabled=use_amp)

    best_val_loss = float("inf")
    best_state = None
    no_improve = 0

    for epoch in range(NUM_EPOCHS_BASELINE):
        model.train()
        optimizer.zero_grad()
        total_loss = 0.0

        for i, batch in enumerate(
            tqdm(train_loader, desc=f"Baseline epoch {epoch+1}/{NUM_EPOCHS_BASELINE}", leave=False)
        ):
            features = get_clip_features(batch["image"], batch["text"])
            labels = batch["label"].to(device)

            with autocast("cuda", enabled=use_amp):
                logits = model(features)
                loss = loss_fn(logits, labels)
                scaled_loss = loss / ACCUM_STEPS

            scaler.scale(scaled_loss).backward()

            is_last = (i + 1) == len(train_loader)

            if (i + 1) % ACCUM_STEPS == 0 or is_last:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()

            total_loss += loss.item()

        val_metrics = evaluate_baseline(model, val_loader)

        print(
            f"Epoch {epoch+1:02d}: "
            f"train_loss={total_loss/max(len(train_loader),1):.4f} | "
            f"val_loss={val_metrics['hate_loss']:.4f} | "
            f"val_acc={val_metrics['overall_acc']:.3f}"
        )

        if val_metrics["hate_loss"] < best_val_loss - 1e-4:
            best_val_loss = val_metrics["hate_loss"]
            best_state = copy.deepcopy(model.state_dict())
            no_improve = 0
            torch.save(best_state, checkpoint_path)
        else:
            no_improve += 1
            if no_improve >= PATIENCE_BASELINE:
                print(f"Baseline early stopping at epoch {epoch+1}.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model

---
# Part 2: Confound-Free 10-Seed Verification Sweep

Everything above this line is unchanged from
`multi3hate_q1_verification_notebook_KAGGLE_FINAL.ipynb`, run through
"9. Training helpers" -- this gives us `train_loader`/`val_loader`/`test_loader`
(the fixed Table I / Run K split), `AdversarialDebiasingModel`, `train_debiased`,
`evaluate_debiased`, and the fixed headline config (`LEARNING_RATE`, `HIDDEN_DIM`,
`LAMBDA_ADV=0.20`, `NUM_EPOCHS_DEBIASED=30`, `PATIENCE_DEBIASED=5`).

Everything below trains 10 independent models at that exact fixed configuration,
varying only the random seed, and verifies each one with `verify_debiasing_run.py`
before trusting its numbers -- resolving the paper's Section 6.3/6.5 limitation
that Run L and Run K could not be cleanly separated from split/budget/environment
confounds. This is now one notebook, one kernel, one continuous run: nothing
needs to be copied between sessions.

**Requires:** the Multi3Hate dataset attached as a Kaggle input (same requirement
as the original notebook above -- this part adds no new external dependency).
**Runtime:** ~10x a single training run (10 seeds x up to 30 epochs each);
budget your Kaggle GPU quota accordingly, or reduce `SEEDS` below for a
faster/partial check.

# 22. Confound-Free Multi-Seed Verification (10 seeds, fixed split/budget)

Resolves the paper's Section 6.3/6.5 open limitation: Run L and Run K in Table I
differ in split construction, training budget, and execution environment, so
seed-driven instability cannot be cleanly separated from those confounds. This
notebook holds everything fixed at Run K's exact configuration and varies **only
the seed**, across 10 seeds instead of 3-5.

**Do NOT need:** any previous results file, checkpoint, or CSV. This cell trains
10 fresh models itself and writes its own output.

**DO need:** append this as a new cell at the end of
`multi3hate_q1_verification_notebook_KAGGLE_FINAL.ipynb`, after running its cells
in order through:
- Cell "0. Check Kaggle inputs and GPU"
- Cell "1. Install only the dependencies needed for this notebook"
- Cell "2. Reproducibility + paths" and "2b. Environment pinning + full determinism"
- Cell "4. Build the corrected real-image + real-caption dataset"
- Cell "5. Reproduce the grouped train/validation/test split" (this fixes the
  split once, using the notebook's global `SEED` -- do NOT re-run this cell
  between the 10 seed iterations below, or the split itself will change)
- Cell "6. Dataset + DataLoader" (creates `train_loader`, `val_loader`, `test_loader`)
- Cell "7. CLIP feature extraction helper"
- Cell "8. Model definitions used by the original final comparison" (defines
  `AdversarialDebiasingModel`, `train_hate_class_weights`, `train_culture_class_weights`)
- Cell "9. Training helpers" (defines `train_debiased`, `evaluate_debiased`,
  the global constants `LEARNING_RATE`, `HIDDEN_DIM`, `LAMBDA_ADV`,
  `NUM_EPOCHS_DEBIASED`, `PATIENCE_DEBIASED`)

You do **not** need to run cell 10 ("Load existing debiased checkpoint...") --
this notebook trains its own 10 checkpoints from scratch, separately, so it
will not overwrite `DEBIAS_CHECKPOINT`.

Also place `verify_debiasing_run.py` in the same working directory first.

## Step 0: write the verification module into this Kaggle kernel

Run this cell FIRST. It writes `verify_debiasing_run.py` into the current
working directory using Jupyter's `%%writefile` magic, so the next cell's
`from verify_debiasing_run import VerificationProtocol` finds it -- no upload
or Kaggle dataset attachment needed. Kaggle's working directory is normally
writable (`/kaggle/working`), so this should work as-is; if `%%writefile`
errors on your setup, run `import os; print(os.getcwd())` and confirm it is
writable, or change the `sys.path.append('.')` line in the next cell to that
path explicitly.

In [13]:
%%writefile verify_debiasing_run.py
"""
verify_debiasing_run.py

A reusable, method-agnostic verification protocol for small-benchmark
fairness/debiasing claims, developed for and validated against the
Multi3Hate cultural-bias adversarial-debiasing project.

Five checks, run in order:
  1. Feature distinctness       (pre-training; catches the data-pipeline defect)
  2. Confusion-matrix degeneracy (catches constant-classifier collapse)
  3. Checkpoint & split identity (catches silent-retrain / stale-checkpoint bugs)
  4. Environment fingerprint     (records, does not gate; for post-hoc diffing)
  5. Bias-gap collapse invariance (Proposition 1: catches a fairness metric that
                                    is an artifact of label prevalence, not fairness)

Usage:
    from verify_debiasing_run import VerificationProtocol, VerificationReport

    report = VerificationProtocol.run(
        model=model,
        test_loader=test_loader,
        feature_fn=lambda batch: get_fused_features(batch),   # pre-training features
        predict_fn=lambda model, batch: model(batch).argmax(-1),
        culture_key="culture",
        label_key="label",
        checkpoint_path=Path("adversarial_debiased_model_best.pt"),
        reference_meme_ids=REFERENCE_TEST_MEME_IDS,           # sorted, deduped, known-good
        prior_bias_gaps=bias_gap_log,                          # running log across your sweep
    )
    report.print_summary()
    if not report.passed:
        raise RuntimeError(f"Verification FAILED: {report.failed_checks}")
"""

from __future__ import annotations

import platform
import subprocess
import sys
from dataclasses import dataclass, field
from pathlib import Path
from typing import Callable, Iterable, Optional

import numpy as np


# --------------------------------------------------------------------------- #
# Result container
# --------------------------------------------------------------------------- #

@dataclass
class CheckResult:
    name: str
    passed: bool
    detail: str
    values: dict = field(default_factory=dict)


@dataclass
class VerificationReport:
    checks: list  # list[CheckResult]

    @property
    def passed(self) -> bool:
        return all(c.passed for c in self.checks)

    @property
    def failed_checks(self) -> list:
        return [c.name for c in self.checks if not c.passed]

    def print_summary(self) -> None:
        print("=" * 70)
        print("VERIFICATION PROTOCOL SUMMARY")
        print("=" * 70)
        for c in self.checks:
            mark = "PASS" if c.passed else "FAIL"
            print(f"[{mark}] {c.name}: {c.detail}")
        print("-" * 70)
        if self.passed:
            print("Overall: SAFE TO REPORT")
        else:
            print(f"Overall: UNTRUSTED -- failed checks: {self.failed_checks}")
        print("=" * 70)


# --------------------------------------------------------------------------- #
# Individual checks
# --------------------------------------------------------------------------- #

def check_feature_distinctness(
    features: np.ndarray,
    cosine_threshold: float = 0.9,
) -> CheckResult:
    """
    Check 1. Run on a batch of fused features BEFORE any model is trained.
    Catches: blank-placeholder images / empty-caption fallbacks (Section 3.2).

    features: (n, d) array of fused image-text embeddings.
    """
    n = features.shape[0]
    n_distinct = np.unique(features, axis=0).shape[0]

    norm = features / np.clip(np.linalg.norm(features, axis=1, keepdims=True), 1e-8, None)
    cos_sim = norm @ norm.T
    off_diag = ~np.eye(n, dtype=bool)
    mean_cos = cos_sim[off_diag].mean()

    ok = (n_distinct == n) and (mean_cos <= cosine_threshold)
    detail = (
        f"{n_distinct}/{n} distinct vectors, mean pairwise cosine sim = {mean_cos:.4f} "
        f"(threshold {cosine_threshold})"
    )
    return CheckResult(
        "1. Feature distinctness", ok, detail,
        values={"n_distinct": n_distinct, "n": n, "mean_cosine_sim": float(mean_cos)},
    )


def check_confusion_matrix_degeneracy(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    precision_eq_accuracy_tol: float = 1e-3,
    recall_zero_tol: float = 1e-3,
) -> CheckResult:
    """
    Check 2. Catches constant-positive and constant-negative collapse
    (Section 4.2; empirically confirmed catching a live failure in Section 5.1.1).
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    tp = np.sum((y_pred == 1) & (y_true == 1))
    fp = np.sum((y_pred == 1) & (y_true == 0))
    fn = np.sum((y_pred == 0) & (y_true == 1))
    tn = np.sum((y_pred == 0) & (y_true == 0))

    accuracy = (tp + tn) / max(len(y_true), 1)
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)

    constant_positive = abs(precision - accuracy) < precision_eq_accuracy_tol and recall > 1 - recall_zero_tol
    constant_negative = recall < recall_zero_tol

    ok = not (constant_positive or constant_negative)
    if constant_positive:
        detail = (
            f"CONSTANT-POSITIVE collapse: precision ({precision:.4f}) == accuracy "
            f"({accuracy:.4f}), recall = {recall:.4f}. fn={fn}, fp={fp} "
            f"(fp should equal the negative-class count if fully collapsed: "
            f"n_negative={tn + fp})"
        )
    elif constant_negative:
        detail = f"CONSTANT-NEGATIVE collapse: recall = {recall:.4f} (tp={tp}, fn={fn})"
    else:
        detail = f"non-degenerate: accuracy={accuracy:.4f}, precision={precision:.4f}, recall={recall:.4f}"

    return CheckResult(
        "2. Confusion-matrix degeneracy", ok, detail,
        values={"tp": int(tp), "fp": int(fp), "fn": int(fn), "tn": int(tn),
                "accuracy": float(accuracy), "precision": float(precision), "recall": float(recall)},
    )


def check_checkpoint_and_split_identity(
    checkpoint_path: Optional[Path],
    checkpoint_loaded_from_disk: bool,
    test_meme_ids: Iterable,
    reference_meme_ids: Iterable,
) -> CheckResult:
    """
    Check 3. Catches silent-retrain fallbacks and split/dtype mismatches
    (Section 4.3; empirically confirmed firing on Run K in this project).
    """
    problems = []

    if checkpoint_path is not None and not checkpoint_loaded_from_disk:
        problems.append(
            f"checkpoint '{checkpoint_path}' was NOT loaded from disk -- this run is a "
            f"freshly-trained model, not the intended saved checkpoint"
        )

    test_ids_sorted = sorted(set(test_meme_ids))
    ref_ids_sorted = sorted(set(reference_meme_ids))
    ids_match = test_ids_sorted == ref_ids_sorted
    if not ids_match:
        missing = set(ref_ids_sorted) - set(test_ids_sorted)
        extra = set(test_ids_sorted) - set(ref_ids_sorted)
        problems.append(f"meme-ID mismatch vs. reference split: missing={len(missing)}, extra={len(extra)}")

    ok = len(problems) == 0
    detail = "checkpoint and split confirmed identical to reference" if ok else "; ".join(problems)
    return CheckResult(
        "3. Checkpoint & split identity", ok, detail,
        values={"n_test_ids": len(test_ids_sorted), "n_reference_ids": len(ref_ids_sorted)},
    )


def record_environment_fingerprint() -> CheckResult:
    """
    Check 4. Always PASSES (it records, it does not gate) -- attach its
    `values` dict to every reported number so environment drift can be
    diffed later (Section 4.4).
    """
    fingerprint = {
        "python": sys.version.split()[0],
        "platform": platform.platform(),
    }
    for pkg in ("torch", "numpy", "pandas", "sklearn", "open_clip"):
        try:
            mod = __import__(pkg)
            fingerprint[pkg] = getattr(mod, "__version__", "unknown")
        except ImportError:
            fingerprint[pkg] = "not installed"

    try:
        freeze = subprocess.run(
            [sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True, timeout=30
        ).stdout
        fingerprint["pip_freeze_lines"] = len(freeze.splitlines())
    except Exception:
        fingerprint["pip_freeze_lines"] = -1

    return CheckResult(
        "4. Environment fingerprint", True,
        f"recorded ({fingerprint.get('python')}, torch={fingerprint.get('torch')})",
        values=fingerprint,
    )


def check_bias_gap_collapse_invariance(
    bias_gap: float,
    confusion_result: CheckResult,
    prior_bias_gaps: Optional[dict] = None,
    sig_figs: int = 4,
    recurrence_threshold: int = 2,
) -> CheckResult:
    """
    Check 5 (new). Implements Proposition 1: if a run's bias gap matches a
    PRIOR run's bias gap to >= sig_figs significant figures, AND either run's
    confusion matrix is degenerate, treat the match as a collapse artifact,
    not a stable fairness result (Section 4.6 / Proposition 1).

    prior_bias_gaps: {config_label: bias_gap_value} accumulated across a sweep.
    Pass the running dict in and this function will also add the current value.
    """
    if prior_bias_gaps is None:
        prior_bias_gaps = {}

    def round_sig(x, sf):
        if x == 0:
            return 0.0
        from math import log10, floor
        return round(x, -int(floor(log10(abs(x)))) + (sf - 1))

    rounded_current = round_sig(bias_gap, sig_figs)
    matches = [label for label, v in prior_bias_gaps.items() if round_sig(v, sig_figs) == rounded_current]

    is_degenerate = not confusion_result.passed
    recurring = len(matches) >= recurrence_threshold - 1  # -1 because current run makes it +1

    ok = not (recurring and is_degenerate)
    if not ok:
        detail = (
            f"bias_gap={bias_gap:.4f} matches {len(matches)} prior run(s) "
            f"({matches}) to {sig_figs} sig figs, AND confusion-matrix check failed "
            f"-- per Proposition 1, this bias gap is a collapse artifact set by test-split "
            f"label prevalence, not a fairness measurement. Re-verify with non-degenerate runs only."
        )
    elif recurring and not is_degenerate:
        detail = (
            f"bias_gap={bias_gap:.4f} recurs across {len(matches)} run(s) but confusion matrix "
            f"is non-degenerate -- recurrence here is more likely genuine, but re-check "
            f"the matching runs' confusion matrices too before trusting this."
        )
    else:
        detail = f"bias_gap={bias_gap:.4f}, no suspicious recurrence detected"

    prior_bias_gaps_out = dict(prior_bias_gaps)
    return CheckResult(
        "5. Bias-gap collapse invariance", ok, detail,
        values={"bias_gap": bias_gap, "matches": matches, "prior_bias_gaps": prior_bias_gaps_out},
    )


# --------------------------------------------------------------------------- #
# Orchestration
# --------------------------------------------------------------------------- #

class VerificationProtocol:
    @staticmethod
    def run(
        *,
        pretrain_features: Optional[np.ndarray] = None,
        y_true: Optional[np.ndarray] = None,
        y_pred: Optional[np.ndarray] = None,
        culture_labels: Optional[np.ndarray] = None,
        checkpoint_path: Optional[Path] = None,
        checkpoint_loaded_from_disk: bool = True,
        test_meme_ids: Optional[Iterable] = None,
        reference_meme_ids: Optional[Iterable] = None,
        prior_bias_gaps: Optional[dict] = None,
    ) -> VerificationReport:
        checks = []

        if pretrain_features is not None:
            checks.append(check_feature_distinctness(pretrain_features))

        confusion_result = None
        if y_true is not None and y_pred is not None:
            confusion_result = check_confusion_matrix_degeneracy(y_true, y_pred)
            checks.append(confusion_result)

        if test_meme_ids is not None and reference_meme_ids is not None:
            checks.append(check_checkpoint_and_split_identity(
                checkpoint_path, checkpoint_loaded_from_disk, test_meme_ids, reference_meme_ids
            ))

        checks.append(record_environment_fingerprint())

        if y_true is not None and y_pred is not None and culture_labels is not None and confusion_result is not None:
            culture_labels = np.asarray(culture_labels)
            y_true_a, y_pred_a = np.asarray(y_true), np.asarray(y_pred)
            per_culture_acc = {
                c: float(np.mean(y_pred_a[culture_labels == c] == y_true_a[culture_labels == c]))
                for c in np.unique(culture_labels)
            }
            bias_gap = max(per_culture_acc.values()) - min(per_culture_acc.values())
            checks.append(check_bias_gap_collapse_invariance(bias_gap, confusion_result, prior_bias_gaps))

        return VerificationReport(checks)


# --------------------------------------------------------------------------- #
# Self-test / demonstration, reproducing Section 5.1.1's Run C finding
# --------------------------------------------------------------------------- #

if __name__ == "__main__":
    # Reproduces the exact Run C degeneracy catch from the paper (Section 5.1.1
    # and Proposition 1): a constant-positive classifier on a 225-example test
    # set (45 per culture) whose per-culture positive prevalence exactly
    # matches the paper's reported values -- so this demo's bias gap should
    # come out to 0.1778, matching the value logged from the zero-shot-CLIP
    # run, and Check 5 should therefore ALSO fire alongside Check 2.
    per_culture_prevalence = {  # from the paper's Table I / Section 5.1.1
        "EN": 25 / 45,   # 0.5556
        "HI": 28 / 45,   # 0.6222
        "DE": 29 / 45,   # 0.6444
        "ZH": 33 / 45,   # 0.7333
        "ES": 31 / 45,   # 0.6889
    }
    y_true_demo, cultures_demo = [], []
    for culture, prevalence in per_culture_prevalence.items():
        n_pos_c = round(prevalence * 45)
        y_true_demo += [1] * n_pos_c + [0] * (45 - n_pos_c)
        cultures_demo += [culture] * 45
    y_true_demo = np.array(y_true_demo)
    cultures_demo = np.array(cultures_demo)
    y_pred_demo = np.ones_like(y_true_demo)  # constant-positive collapse

    report = VerificationProtocol.run(
        y_true=y_true_demo,
        y_pred=y_pred_demo,
        culture_labels=cultures_demo,
        # simulates the running log from a sweep where zero-shot CLIP (a
        # confirmed constant-NEGATIVE collapse) already logged the same
        # bias gap under Proposition 1's opposite-collapse symmetry
        prior_bias_gaps={"zero_shot_clip (constant-negative)": 0.1778},
    )
    report.print_summary()
    print(f"\nComputed bias gap: {max(per_culture_prevalence.values()) - min(per_culture_prevalence.values()):.4f}")
    print("(matches the paper's recurring 0.1778 to 4 significant figures, as Proposition 1 predicts)")


Writing verify_debiasing_run.py


## Step 0.5: Preflight check

Run this before the sweep cell. It checks every variable/function the sweep
needs and prints exactly which of your notebook's setup cells to run for each
one that's missing, instead of stopping at the first `NameError` it happens
to hit (which is what just happened with `MODELS_DIR`).

In [14]:
#@title Step 0.5: Preflight check -- confirms every prerequisite exists before training anything
#@markdown Run this AFTER your KAGGLE_FINAL setup cells and the %%writefile cell
#@markdown above, and BEFORE the sweep cell below. It checks every name the
#@markdown sweep cell needs and tells you exactly which setup cell to (re)run
#@markdown for each missing one, instead of crashing partway through training.

# Attempt the import ourselves -- the %%writefile cell above only writes
# verify_debiasing_run.py to disk, it does not import it. This is the first
# cell that actually needs VerificationProtocol, so it imports it here.
try:
    from verify_debiasing_run import VerificationProtocol
except ImportError as e:
    print(f"Could not import verify_debiasing_run yet: {e}")
    print("Make sure the %%writefile verify_debiasing_run.py cell above has been run.")

_required = {
    "BASE_DIR":                   'Cell "2. Reproducibility + paths"',
    "MODELS_DIR":                 'Cell "2. Reproducibility + paths"',
    "RESULTS_DIR":                'Cell "2. Reproducibility + paths"',
    "SEED":                       'Cell "2. Reproducibility + paths"',
    "device":                     'Cell "2. Reproducibility + paths"',
    "train_loader":               'Cell "6. Dataset + DataLoader"',
    "val_loader":                 'Cell "6. Dataset + DataLoader"',
    "test_loader":                'Cell "6. Dataset + DataLoader"',
    "GLOBAL_CULTURE_TO_IDX":      'Cell "4. Build the corrected real-image + real-caption dataset" '
                                   '(or wherever your notebook defines this mapping -- search for it '
                                   'if this check fails and it is not in cell 4)',
    "FUSED_FEATURE_DIM":          'Cell "6. Dataset + DataLoader"',
    "get_clip_features":          'Cell "7. CLIP feature extraction helper"',
    "AdversarialDebiasingModel":  'Cell "8. Model definitions used by the original final comparison"',
    "train_hate_class_weights":   'Cell "8. Model definitions used by the original final comparison"',
    "train_culture_class_weights": 'Cell "8. Model definitions used by the original final comparison"',
    "train_debiased":             'Cell "9. Training helpers"',
    "evaluate_debiased":          'Cell "9. Training helpers"',
    "LEARNING_RATE":              'Cell "9. Training helpers"',
    "HIDDEN_DIM":                 'Cell "9. Training helpers"',
    "LAMBDA_ADV":                 'Cell "9. Training helpers"',
    "NUM_EPOCHS_DEBIASED":        'Cell "9. Training helpers"',
    "PATIENCE_DEBIASED":          'Cell "9. Training helpers"',
    "VerificationProtocol":       "the %%writefile verify_debiasing_run.py cell above (re-run it, "
                                   "then re-run this check)",
}

_missing = {name: where for name, where in _required.items() if name not in globals()}

if _missing:
    print("=" * 70)
    print(f"PREFLIGHT FAILED -- {len(_missing)} of {len(_required)} required names are missing")
    print("=" * 70)
    for name, where in _missing.items():
        print(f"  MISSING: {name:<32} -> run {where}")
    print("\nRun the cells listed above (in the order they appear in your original")
    print("notebook), then re-run this preflight cell before the sweep cell.")
    raise NameError(
        f"{len(_missing)} prerequisite(s) not yet defined in this kernel: {list(_missing)}. "
        f"See the printed list above for which cell to run."
    )
else:
    print("=" * 70)
    print(f"PREFLIGHT PASSED -- all {len(_required)} required names are defined.")
    print("Safe to run the sweep cell now.")
    print("=" * 70)


PREFLIGHT PASSED -- all 22 required names are defined.
Safe to run the sweep cell now.


In [ ]:
#@title 22. CONFOUND-FREE Multi-Seed Verification (10 seeds, fixed split/budget)
#@markdown Uses the notebook's REAL train_debiased()/evaluate_debiased() API and
#@markdown global constants (LEARNING_RATE, HIDDEN_DIM, LAMBDA_ADV,
#@markdown NUM_EPOCHS_DEBIASED, PATIENCE_DEBIASED) exactly as defined in
#@markdown Cell 9 -- nothing about the training config changes except the seed.
#@markdown Every seed is checked with verify_debiasing_run.py's Checks 1/2/5
#@markdown before being trusted, so a Run-C-style collapse cannot silently
#@markdown enter the summary table.

import sys, random, copy
sys.path.append('.')
from verify_debiasing_run import VerificationProtocol

N_SEEDS = 10
SEEDS = [42, 123, 2024, 7, 2025, 13, 99, 512, 777, 31415]  # first 5 match Table VIII

SWEEP_DIR = MODELS_DIR / "confound_free_sweep"
SWEEP_DIR.mkdir(parents=True, exist_ok=True)


def evaluate_debiased_with_cultures(model, loader):
    """Identical to Cell 9's evaluate_debiased(), plus raw per-example culture
    labels (needed for verify_debiasing_run's Check 5 / Proposition 1)."""
    model.eval()
    all_preds, all_labels, all_cultures = [], [], []
    total_loss, total_adv_loss = 0.0, 0.0

    hate_loss_fn = nn.CrossEntropyLoss(weight=train_hate_class_weights.to(device))
    culture_loss_fn = nn.CrossEntropyLoss(weight=train_culture_class_weights.to(device))

    for batch in tqdm(loader, desc="Evaluating (sweep)", leave=False):
        features = get_clip_features(batch["image"], batch["text"])
        labels = batch["label"].to(device)
        cultures = batch["culture_idx"].to(device)

        hate_logits, culture_logits = model(features)
        total_loss += hate_loss_fn(hate_logits, labels).item()
        total_adv_loss += culture_loss_fn(culture_logits, cultures).item()

        all_preds.extend(hate_logits.argmax(dim=1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_cultures.extend(batch["culture"])

    culture_results = {}
    for culture in sorted(set(all_cultures)):
        mask = np.array(all_cultures) == culture
        culture_results[culture] = accuracy_score(
            np.array(all_labels)[mask], np.array(all_preds)[mask]
        )

    return {
        "hate_loss": total_loss / max(len(loader), 1),
        "adv_loss": total_adv_loss / max(len(loader), 1),
        "hate_acc": accuracy_score(all_labels, all_preds),
        "culture_results": culture_results,
        "predictions": all_preds,
        "labels": all_labels,
        "cultures": all_cultures,
    }


sweep_rows = []
prior_bias_gap_log = {}  # accumulates across this sweep for Check 5 (Proposition 1)

print(f"Running {N_SEEDS} seeds at Run K's FIXED config: LR={LEARNING_RATE}, "
      f"HIDDEN_DIM={HIDDEN_DIM}, LAMBDA_ADV={LAMBDA_ADV}, "
      f"epochs={NUM_EPOCHS_DEBIASED}, patience={PATIENCE_DEBIASED}")
print("(train_loader/val_loader/test_loader are the FIXED Cell-5 split, unchanged across seeds)\n")

for seed in SEEDS:
    print(f"\n{'='*60}\nSEED {seed}\n{'='*60}")

    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    model = AdversarialDebiasingModel(
        feature_dim=FUSED_FEATURE_DIM,
        num_classes=2,
        num_cultures=len(GLOBAL_CULTURE_TO_IDX),
        hidden_dim=HIDDEN_DIM,
        lambda_adv=LAMBDA_ADV,
    ).to(device)

    ckpt_path = SWEEP_DIR / f"debiased_seed_{seed}.pt"
    model, history = train_debiased(model, train_loader, val_loader, ckpt_path)
    model.eval()

    metrics = evaluate_debiased_with_cultures(model, test_loader)
    y_true = np.array(metrics["labels"])
    y_pred = np.array(metrics["predictions"])
    cultures = np.array(metrics["cultures"])

    report = VerificationProtocol.run(
        y_true=y_true, y_pred=y_pred, culture_labels=cultures,
        prior_bias_gaps=dict(prior_bias_gap_log),
    )

    culture_accs = list(metrics["culture_results"].values())
    bias_gap = max(culture_accs) - min(culture_accs)
    prior_bias_gap_log[f"seed_{seed}"] = bias_gap

    sweep_rows.append({
        "seed": seed,
        "hate_acc": metrics["hate_acc"],
        "bias_gap": bias_gap,
        "verification_passed": report.passed,
        "failed_checks": ";".join(report.failed_checks) if not report.passed else "",
    })

    status = "PASS" if report.passed else f"FAIL ({report.failed_checks})"
    print(f"  hate_acc={metrics['hate_acc']:.4f}  bias_gap={bias_gap:.4f}  verification={status}")
    print(f"  per-culture: {{k: round(v, 4) for k, v in metrics['culture_results'].items()}}")

sweep_df = pd.DataFrame(sweep_rows)
print("\n" + "=" * 70)
print("CONFOUND-FREE 10-SEED SWEEP -- FULL TABLE (Run K's fixed split/budget/config)")
print("=" * 70)
display(sweep_df)

trusted = sweep_df[sweep_df["verification_passed"]]
untrusted = sweep_df[~sweep_df["verification_passed"]]

print(f"\nSeeds passing verification: {len(trusted)}/{N_SEEDS}")
print(f"Seeds FAILING verification (excluded from summary stats): {len(untrusted)}/{N_SEEDS}")
if len(untrusted) > 0:
    print("  -- report the failure rate itself, exactly as Run C's collapse was "
          "reported in Section 5.1.1, not silently dropped.")

if len(trusted) > 0:
    print("\nSummary over VERIFIED (non-degenerate) seeds only:")
    print(f"  hate_acc : {trusted['hate_acc'].mean():.4f} +/- {trusted['hate_acc'].std():.4f}")
    print(f"  bias_gap : {trusted['bias_gap'].mean():.4f} +/- {trusted['bias_gap'].std():.4f}")
    print(f"  range    : hate_acc [{trusted['hate_acc'].min():.4f}, {trusted['hate_acc'].max():.4f}], "
          f"bias_gap [{trusted['bias_gap'].min():.4f}, {trusted['bias_gap'].max():.4f}]")

collapse_rate = len(untrusted) / N_SEEDS
print(f"\nCOLLAPSE / VERIFICATION-FAILURE RATE at Run K's exact configuration: {collapse_rate:.1%}")
print("This is now directly comparable to Table I and Table VIII, since split, "
      "budget, and config are identical to the headline Run K -- only seed varies.")

sweep_csv = RESULTS_DIR / "confound_free_10seed_sweep.csv"
sweep_df.to_csv(sweep_csv, index=False)
print(f"\nSaved: {sweep_csv}")


Running 10 seeds at Run K's FIXED config: LR=0.0001, HIDDEN_DIM=512, LAMBDA_ADV=0.2, epochs=30, patience=5
(train_loader/val_loader/test_loader are the FIXED Cell-5 split, unchanged across seeds)


SEED 42


Debiased epoch 1/30:   0%|          | 0/66 [00:00<?, ?it/s]

Evaluating debiased:   0%|          | 0/15 [00:00<?, ?it/s]

Epoch 01: train_hate=0.6928 | train_adv=1.6057 | val_hate_loss=0.6930 | val_acc=0.578 | lambda=0.0000


Debiased epoch 2/30:   0%|          | 0/66 [00:00<?, ?it/s]

...

Debiased epoch 1/30:   0%|          | 0/66 [00:00<?, ?it/s]

Evaluating debiased:   0%|          | 0/15 [00:00<?, ?it/s]

Epoch 01: train_hate=0.6930 | train_adv=1.6076 | val_hate_loss=0.6929 | val_acc=0.556 | lambda=0.0000


Debiased epoch 2/30:   0%|          | 0/66 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7decec18b600>
 Exception ignored in: Traceback (most recent call last):
<function _MultiProcessingDataLoaderIter.__del__ at 0x7decec18b600>  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__

    Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
self._shutdown_workers()    
self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
      File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
if w.is_alive():    if w.is_alive():
Exception ignored in:  
<function _MultiProcessingDataLoaderIter.__del__ at 0x7decec18b600>  
  Traceback (most recent call last):
    File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line

## Step 3: Finish the sweep (recover from the SEED 31415 interruption)

The previous run of the sweep cell above completed 9 of 10 seeds before being
cut off (Kaggle session interruption during seed 31415, with no error -- the
pattern of a clean stop with no traceback is consistent with a session
timeout rather than a code bug). This cell hardcodes the 9 already-verified
results and runs only the missing seed, then produces the complete, merged
10-seed summary and CSV.

If your kernel session survived the interruption, this cell needs nothing
extra. If the session was fully reset, re-run the setup chain and the
`%%writefile verify_debiasing_run.py` cell above first (the preflight check
will tell you if anything else is missing).

In [17]:
#@title Step 3: Finish the sweep -- run only the missing seed and merge
#@markdown The previous run completed 9/10 seeds before being interrupted
#@markdown (Kaggle session timeout during SEED 31415). This cell hardcodes
#@markdown the 9 already-verified results, runs ONLY seed 31415 fresh, and
#@markdown produces the complete 10-seed summary -- no need to redo the 9
#@markdown that already finished.
#@markdown
#@markdown Requires the same prerequisites as the main sweep cell (run the
#@markdown preflight check first if you are not sure they are still defined
#@markdown in this kernel -- if your session was fully reset, you will need
#@markdown to re-run the setup chain and the %%writefile cell again first).

import sys, random
sys.path.append('.')
from verify_debiasing_run import VerificationProtocol

# --- The 9 seeds that already completed and passed verification ---
ALREADY_DONE = [
    {"seed": 42,    "hate_acc": 0.6044, "bias_gap": 0.0667, "verification_passed": True, "failed_checks": ""},
    {"seed": 123,   "hate_acc": 0.6800, "bias_gap": 0.1556, "verification_passed": True, "failed_checks": ""},
    {"seed": 2024,  "hate_acc": 0.6578, "bias_gap": 0.1556, "verification_passed": True, "failed_checks": ""},
    {"seed": 7,     "hate_acc": 0.6400, "bias_gap": 0.1333, "verification_passed": True, "failed_checks": ""},
    {"seed": 2025,  "hate_acc": 0.6800, "bias_gap": 0.1556, "verification_passed": True, "failed_checks": ""},
    {"seed": 13,    "hate_acc": 0.6667, "bias_gap": 0.1333, "verification_passed": True, "failed_checks": ""},
    {"seed": 99,    "hate_acc": 0.6711, "bias_gap": 0.1556, "verification_passed": True, "failed_checks": ""},
    {"seed": 512,   "hate_acc": 0.6800, "bias_gap": 0.1556, "verification_passed": True, "failed_checks": ""},
    {"seed": 777,   "hate_acc": 0.6756, "bias_gap": 0.1333, "verification_passed": True, "failed_checks": ""},
]
MISSING_SEED = 31415

SWEEP_DIR = MODELS_DIR / "confound_free_sweep"
SWEEP_DIR.mkdir(parents=True, exist_ok=True)


def evaluate_debiased_with_cultures(model, loader):
    """Same as in the main sweep cell -- returns raw per-example culture
    labels alongside the standard evaluate_debiased() metrics."""
    model.eval()
    all_preds, all_labels, all_cultures = [], [], []
    total_loss, total_adv_loss = 0.0, 0.0

    hate_loss_fn = nn.CrossEntropyLoss(weight=train_hate_class_weights.to(device))
    culture_loss_fn = nn.CrossEntropyLoss(weight=train_culture_class_weights.to(device))

    for batch in tqdm(loader, desc="Evaluating (finish sweep)", leave=False):
        features = get_clip_features(batch["image"], batch["text"])
        labels = batch["label"].to(device)
        cultures = batch["culture_idx"].to(device)

        hate_logits, culture_logits = model(features)
        total_loss += hate_loss_fn(hate_logits, labels).item()
        total_adv_loss += culture_loss_fn(culture_logits, cultures).item()

        all_preds.extend(hate_logits.argmax(dim=1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_cultures.extend(batch["culture"])

    culture_results = {}
    for culture in sorted(set(all_cultures)):
        mask = np.array(all_cultures) == culture
        culture_results[culture] = accuracy_score(
            np.array(all_labels)[mask], np.array(all_preds)[mask]
        )

    return {
        "hate_loss": total_loss / max(len(loader), 1),
        "adv_loss": total_adv_loss / max(len(loader), 1),
        "hate_acc": accuracy_score(all_labels, all_preds),
        "culture_results": culture_results,
        "predictions": all_preds,
        "labels": all_labels,
        "cultures": all_cultures,
    }


# --- Run only the missing seed ---
print(f"Running the one missing seed: {MISSING_SEED}")
print(f"Config: LR={LEARNING_RATE}, HIDDEN_DIM={HIDDEN_DIM}, LAMBDA_ADV={LAMBDA_ADV}, "
      f"epochs={NUM_EPOCHS_DEBIASED}, patience={PATIENCE_DEBIASED}\n")

torch.manual_seed(MISSING_SEED)
np.random.seed(MISSING_SEED)
random.seed(MISSING_SEED)

model = AdversarialDebiasingModel(
    feature_dim=FUSED_FEATURE_DIM,
    num_classes=2,
    num_cultures=len(GLOBAL_CULTURE_TO_IDX),
    hidden_dim=HIDDEN_DIM,
    lambda_adv=LAMBDA_ADV,
).to(device)

ckpt_path = SWEEP_DIR / f"debiased_seed_{MISSING_SEED}.pt"
model, history = train_debiased(model, train_loader, val_loader, ckpt_path)
model.eval()

metrics = evaluate_debiased_with_cultures(model, test_loader)
y_true = np.array(metrics["labels"])
y_pred = np.array(metrics["predictions"])
cultures = np.array(metrics["cultures"])

# Prior bias gaps from the 9 completed seeds, for the Check-5 comparison
prior_bias_gap_log = {f"seed_{r['seed']}": r["bias_gap"] for r in ALREADY_DONE}

report = VerificationProtocol.run(
    y_true=y_true, y_pred=y_pred, culture_labels=cultures,
    prior_bias_gaps=prior_bias_gap_log,
)

culture_accs = list(metrics["culture_results"].values())
bias_gap = max(culture_accs) - min(culture_accs)

status = "PASS" if report.passed else f"FAIL ({report.failed_checks})"
print(f"  hate_acc={metrics['hate_acc']:.4f}  bias_gap={bias_gap:.4f}  verification={status}")
print(f"  per-culture: {{k: round(v, 4) for k, v in metrics['culture_results'].items()}}")

# --- Merge into the complete 10-seed table ---
sweep_rows = list(ALREADY_DONE) + [{
    "seed": MISSING_SEED,
    "hate_acc": metrics["hate_acc"],
    "bias_gap": bias_gap,
    "verification_passed": report.passed,
    "failed_checks": ";".join(report.failed_checks) if not report.passed else "",
}]

sweep_df = pd.DataFrame(sweep_rows)
print("\n" + "=" * 70)
print("COMPLETE 10-SEED SWEEP -- FULL TABLE (Run K's fixed split/budget/config)")
print("=" * 70)
display(sweep_df)

trusted = sweep_df[sweep_df["verification_passed"]]
untrusted = sweep_df[~sweep_df["verification_passed"]]

print(f"\nSeeds passing verification: {len(trusted)}/{len(sweep_df)}")
print(f"Seeds FAILING verification (excluded from summary stats): {len(untrusted)}/{len(sweep_df)}")
if len(untrusted) > 0:
    print("  -- report the failure rate itself, exactly as Run C's collapse was "
          "reported in Section 5.1.1, not silently dropped.")

if len(trusted) > 0:
    print("\nSummary over VERIFIED (non-degenerate) seeds only:")
    print(f"  hate_acc : {trusted['hate_acc'].mean():.4f} +/- {trusted['hate_acc'].std():.4f}")
    print(f"  bias_gap : {trusted['bias_gap'].mean():.4f} +/- {trusted['bias_gap'].std():.4f}")
    print(f"  range    : hate_acc [{trusted['hate_acc'].min():.4f}, {trusted['hate_acc'].max():.4f}], "
          f"bias_gap [{trusted['bias_gap'].min():.4f}, {trusted['bias_gap'].max():.4f}]")

collapse_rate = len(untrusted) / len(sweep_df)
print(f"\nCOLLAPSE / VERIFICATION-FAILURE RATE at Run K's exact configuration: {collapse_rate:.1%}")

sweep_csv = RESULTS_DIR / "confound_free_10seed_sweep.csv"
sweep_df.to_csv(sweep_csv, index=False)
print(f"\nSaved: {sweep_csv}")


Running the one missing seed: 31415
Config: LR=0.0001, HIDDEN_DIM=512, LAMBDA_ADV=0.2, epochs=30, patience=5



Debiased epoch 1/30:   0%|          | 0/66 [00:00<?, ?it/s]

Evaluating debiased:   0%|          | 0/15 [00:00<?, ?it/s]

Epoch 01: train_hate=0.6930 | train_adv=1.6054 | val_hate_loss=0.6932 | val_acc=0.609 | lambda=0.0000


Debiased epoch 2/30:   0%|          | 0/66 [00:00<?, ?it/s]

...

Epoch 13: train_hate=0.4898 | train_adv=1.5714 | val_hate_loss=0.6948 | val_acc=0.622 | lambda=0.1937
Early stopping at epoch 13.


Evaluating (finish sweep):   0%|          | 0/15 [00:00<?, ?it/s]

  hate_acc=0.6533  bias_gap=0.1556  verification=PASS
  per-culture: {k: round(v, 4) for k, v in metrics['culture_results'].items()}

COMPLETE 10-SEED SWEEP -- FULL TABLE (Run K's fixed split/budget/config)


,seed,hate_acc,bias_gap,verification_passed,failed_checks
0,42,0.604400,0.066700,True,
1,123,0.680000,0.155600,True,
2,2024,0.657800,0.155600,True,
3,7,0.640000,0.133300,True,
4,2025,0.680000,0.155600,True,
5,13,0.666700,0.133300,True,
6,99,0.671100,0.155600,True,
7,512,0.680000,0.155600,True,
8,777,0.675600,0.133300,True,
9,31415,0.653333,0.155556,True,



Seeds passing verification: 10/10
Seeds FAILING verification (excluded from summary stats): 0/10

Summary over VERIFIED (non-degenerate) seeds only:
  hate_acc : 0.6609 +/- 0.0239
  bias_gap : 0.1400 +/- 0.0278
  range    : hate_acc [0.6044, 0.6800], bias_gap [0.0667, 0.1556]

COLLAPSE / VERIFICATION-FAILURE RATE at Run K's exact configuration: 0.0%

Saved: /kaggle/working/Multi3Hate/results/confound_free_10seed_sweep.csv
